# Análise do TCC — Importações de NCM 29 (2023–2025)


## 1. Configurações

In [1]:
from pathlib import Path
import gc
import re
import zipfile
import unicodedata
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import requests
from scipy import stats
from scipy.cluster.hierarchy import linkage, cut_tree, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

PASTA = Path('/content/tcc')
DADOS = PASTA / 'dados'
SAIDA = PASTA / 'resultados'
FIG = SAIDA / 'figuras'
EXTRAS = SAIDA / 'graficos_complementares'
for pasta in (DADOS, SAIDA, FIG, EXTRAS):
    pasta.mkdir(parents=True, exist_ok=True)

ANOS = (2023, 2024, 2025)
MIN_OBS = 10
R2_MIN = 0.70
# Faixa exploratória: o mínimo de três grupos busca permitir subdivisões
# que possam refletir diferenciações entre produtos; não afirma classes naturais.
K_MIN_GRUPOS = 3
K_MAX_GRUPOS = 10
K_CANDIDATOS = range(K_MIN_GRUPOS, K_MAX_GRUPOS + 1)
SEMENTE = 42
RF_ARVORES = 100


CPI = {2023: 304.702, 2024: 313.689, 2025: 321.943}
FONTE_CPI = 'https://www.bls.gov/regions/mid-atlantic/data/consumerpriceindexannualandsemiannual_table.htm'


EXEMPLOS = {
    '29051100': 'Metanol',
    '29071100': 'Fenol',
    '29032200': 'Tricloroetileno',
    '29153932': 'Acetato de 2-etoxietila',
}

FONTES = {
    **{f'IMP_{ano}.csv': f'https://balanca.mdic.gov.br/balanca/bd/comexstat-bd/ncm/IMP_{ano}.csv' for ano in ANOS},
    'NCM.csv': 'https://balanca.mdic.gov.br/balanca/bd/tabelas/NCM.csv',
}

plt.rcParams.update({'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 110})
print('Pasta de trabalho:', PASTA)

Pasta de trabalho: /content/tcc


## 2. Baixar os dados do MDIC



In [2]:
import time
import warnings
from urllib.parse import urlparse

import requests
from urllib3.exceptions import InsecureRequestWarning


PERMITIR_SSL_SEM_VALIDACAO_MDIC = True


def conferir_csv(caminho, nome):
    if not caminho.exists() or caminho.stat().st_size < 1000:
        return False
    with caminho.open('rb') as f:
        cabecalho = f.readline(2048).decode('utf-8-sig', errors='replace').upper()
    if nome.startswith('IMP_'):
        campos = ('CO_ANO', 'CO_NCM', 'KG_LIQUIDO', 'VL_FOB')
    else:
        campos = ('CO_NCM', 'NO_NCM')
    return all(campo in cabecalho for campo in campos)


def obter_arquivo(nome, url, tentativas=5):
    endereco = urlparse(url)
    if endereco.scheme != 'https' or endereco.hostname != 'balanca.mdic.gov.br':
        raise ValueError('O endereço não pertence ao servidor oficial esperado: ' + url)

    destino = DADOS / nome
    parcial = DADOS / (nome + '.parcial')
    if conferir_csv(destino, nome):
        print(f'Usando arquivo existente: {nome} ({destino.stat().st_size/1024**2:.1f} MB)')
        return destino
    if destino.exists():
        destino.unlink()

    verificar_ssl = True
    ultimo_erro = None
    print(f'Baixando {nome} ...', flush=True)

    for tentativa in range(1, tentativas + 1):
        tamanho_parcial = parcial.stat().st_size if parcial.exists() else 0
        headers = {
            'User-Agent': 'Mozilla/5.0 (pesquisa-academica; ComexStat)',
            'Accept-Encoding': 'identity',
        }
        if tamanho_parcial:
            headers['Range'] = f'bytes={tamanho_parcial}-'

        try:

            try:
                resposta = requests.get(
                    url, headers=headers, stream=True, allow_redirects=True,
                    timeout=(30, 120), verify=verificar_ssl
                )
            except requests.exceptions.SSLError:
                if not (verificar_ssl and PERMITIR_SSL_SEM_VALIDACAO_MDIC):
                    raise
                print('  Certificado HTTPS rejeitado pelo Colab.')
                verificar_ssl = False
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore', InsecureRequestWarning)
                    resposta = requests.get(
                        url, headers=headers, stream=True, allow_redirects=True,
                        timeout=(30, 120), verify=False
                    )

            with resposta:
                final = urlparse(resposta.url)
                if final.scheme != 'https' or not (
                    final.hostname == 'balanca.mdic.gov.br' or
                    final.hostname.endswith('.mdic.gov.br')
                ):
                    raise RuntimeError('Redirecionamento fora do domínio HTTPS do MDIC: ' + resposta.url)

                if resposta.status_code == 416:
                    total_servidor = resposta.headers.get('Content-Range', '').split('/')[-1]
                    if (total_servidor.isdigit() and int(total_servidor) == tamanho_parcial
                            and conferir_csv(parcial, nome)):
                        parcial.replace(destino)
                        print(f'  {nome}: concluído (arquivo parcial já estava completo).')
                        return destino

                resposta.raise_for_status()

                if resposta.status_code == 206 and tamanho_parcial:
                    cr = resposta.headers.get('Content-Range', '')
                    if not cr.startswith(f'bytes {tamanho_parcial}-'):
                        raise RuntimeError('Retomada inválida informada pelo servidor: ' + cr)
                    modo = 'ab'
                    recebidos = tamanho_parcial
                else:
                    modo = 'wb'
                    recebidos = 0

                tamanho_servidor = resposta.headers.get('Content-Length', '')
                esperado = (recebidos + int(tamanho_servidor)) if tamanho_servidor.isdigit() else None
                proximo_aviso = (recebidos // (25*1024**2) + 1) * 25*1024**2

                with parcial.open(modo) as arquivo:
                    for bloco in resposta.iter_content(chunk_size=1024*1024):
                        if not bloco:
                            continue
                        arquivo.write(bloco)
                        recebidos += len(bloco)
                        if recebidos >= proximo_aviso:
                            print(f'  {nome}: {recebidos/1024**2:.0f} MB', flush=True)
                            proximo_aviso += 25*1024**2

                if esperado is not None and recebidos != esperado:
                    raise IOError(f'Download interrompido: {recebidos} de {esperado} bytes.')

            if not conferir_csv(parcial, nome):
                raise ValueError('Conteúdo recebido não parece ser o CSV oficial.')

            parcial.replace(destino)
            print(f'  Concluído: {nome} ({destino.stat().st_size/1024**2:.1f} MB)', flush=True)
            return destino

        except (requests.RequestException, OSError, ValueError, RuntimeError) as erro:
            ultimo_erro = erro
            print(f'  Tentativa {tentativa}/{tentativas}: {type(erro).__name__}: {str(erro)[:140]}', flush=True)
            if tentativa < tentativas:
                time.sleep(min(tentativa * 2, 8))

    raise RuntimeError(
        f'Não foi possível obter {nome}. Último erro: '
        f'{type(ultimo_erro).__name__}: {str(ultimo_erro)[:180]}. '
        'Os arquivos parciais foram preservados. Se o servidor não responder, '
        'use a célula opcional de envio manual.'
    )


for nome, url in FONTES.items():
    obter_arquivo(nome, url)
print('Os quatro arquivos estão disponíveis. Pode prosseguir para as análises.')


Baixando IMP_2023.csv ...
  Certificado HTTPS rejeitado pelo Colab.
  IMP_2023.csv: 25 MB
  IMP_2023.csv: 50 MB
  IMP_2023.csv: 75 MB
  IMP_2023.csv: 100 MB
  IMP_2023.csv: 125 MB
  Concluído: IMP_2023.csv (145.9 MB)
Baixando IMP_2024.csv ...
  Certificado HTTPS rejeitado pelo Colab.
  IMP_2024.csv: 25 MB
  IMP_2024.csv: 50 MB
  IMP_2024.csv: 75 MB
  IMP_2024.csv: 100 MB
  IMP_2024.csv: 125 MB
  IMP_2024.csv: 150 MB
  Concluído: IMP_2024.csv (159.0 MB)
Baixando IMP_2025.csv ...
  Certificado HTTPS rejeitado pelo Colab.
  IMP_2025.csv: 25 MB
  IMP_2025.csv: 50 MB
  IMP_2025.csv: 75 MB
  IMP_2025.csv: 100 MB
  IMP_2025.csv: 125 MB
  IMP_2025.csv: 150 MB
  Concluído: IMP_2025.csv (167.4 MB)
Baixando NCM.csv ...
  Certificado HTTPS rejeitado pelo Colab.
  Concluído: NCM.csv (3.2 MB)
Os quatro arquivos estão disponíveis. Pode prosseguir para as análises.


### Alternativa manual (somente se o servidor MDIC estiver indisponível)


In [ ]:
if False:  # troque por True apenas para enviar manualmente
    from google.colab import files
    enviados = files.upload()
    for nome, conteudo in enviados.items():
        destino = DADOS / Path(nome).name
        destino.write_bytes(conteudo)
        if not conferir_csv(destino, destino.name):
            raise ValueError(f'{nome} não contém o cabeçalho CSV esperado.')
        print('Recebido:', destino)
    del enviados
    gc.collect()


## 3. Preparar o dicionário e os registros

In [3]:
SUBCAPITULOS = [
    (2901, 2904, 'I'), (2905, 2906, 'II'), (2907, 2908, 'III'),
    (2909, 2911, 'IV'), (2912, 2913, 'V'), (2914, 2914, 'VI'),
    (2915, 2918, 'VII'), (2919, 2920, 'VIII'), (2921, 2929, 'IX'),
    (2930, 2935, 'X'), (2936, 2937, 'XI'), (2938, 2939, 'XII'),
    (2940, 2942, 'XIII'),
]

def subcapitulo(ncm):
    pos = int(str(ncm)[:4])
    return next((codigo for a, b, codigo in SUBCAPITULOS if a <= pos <= b), '')

def nome_coluna(s):
    s = unicodedata.normalize('NFKD', str(s))
    return re.sub('[^A-Z0-9]+', '_', s.encode('ascii', 'ignore').decode().upper()).strip('_')

def ler_dicionario(arquivo):
    tab = None
    for enc in ('utf-8-sig', 'latin-1'):
        try:
            tab = pd.read_csv(arquivo, sep=';', dtype=str, encoding=enc, engine='python')
            if len(tab.columns) >= 2:
                break
        except (UnicodeError, pd.errors.ParserError):
            tab = None
    if tab is None:
        texto = arquivo.read_bytes().decode('utf-8-sig', errors='replace')
        if texto.count('\ufffd') > 10:
            texto = arquivo.read_bytes().decode('latin-1')
        linhas = texto.splitlines()
        colunas = [c.strip(' "\ufeff') for c in linhas[0].split(';')]
        blocos, atual = [], ''
        for linha in linhas[1:]:
            if re.match(r'^\s*"?\d{8}"?\s*;', linha):
                if atual:
                    blocos.append(atual)
                atual = linha
            elif atual:
                atual += ' ' + linha
        if atual:
            blocos.append(atual)
        import csv
        registros = []
        for bloco in blocos:
            try:
                campos = next(csv.reader([bloco], delimiter=';'))
            except csv.Error:
                campos = bloco.split(';')
            if len(campos) == len(colunas):
                registros.append(campos)
            elif bloco.lstrip(' "').startswith('29'):
                raise ValueError('Registro NCM 29 com formato inválido. Confira NCM.csv.')
        tab = pd.DataFrame(registros, columns=colunas)
    col = {nome_coluna(c): c for c in tab.columns}
    chave = next((col[x] for x in ['CO_NCM', 'NCM', 'CODIGO_NCM'] if x in col), None)
    texto = next((col[x] for x in ['NO_NCM_POR', 'DESCRICAO_NCM', 'DESCRICAO', 'NO_NCM'] if x in col), None)
    if not chave or not texto:
        raise ValueError(f'O dicionário precisa das colunas de código e descrição da NCM: {list(tab.columns)}')
    cod = tab[chave].astype(str).str.replace(r'\D', '', regex=True).str.zfill(8)
    descricao = tab[texto].fillna('').astype(str).str.strip()
    base = pd.DataFrame({'ncm': cod, 'descricao_ncm': descricao})
    base = base[base.ncm.str.fullmatch(r'29\d{6}') & base.descricao_ncm.ne('')]
    if base.groupby('ncm').descricao_ncm.nunique().max() > 1:
        raise ValueError('Há descrições conflitantes no dicionário. Confira NCM.csv.')
    base = base.drop_duplicates('ncm')
    base.to_csv(SAIDA / 'dicionario_ncm.csv', index=False, encoding='utf-8-sig')
    return base.set_index('ncm').descricao_ncm.to_dict()

DESCRICOES = ler_dicionario(DADOS / 'NCM.csv')
print('Descrições de NCMs do capítulo 29:', len(DESCRICOES))

COLUNAS = ['CO_ANO', 'CO_MES', 'CO_NCM', 'KG_LIQUIDO', 'VL_FOB']

def ler_importacoes(ano):
    arquivo = DADOS / f'IMP_{ano}.csv'
    partes = []
    codigos_iniciais = set()
    n_validos = 0
    meses = set()
    n_brutos = 0
    for bloco in pd.read_csv(arquivo, sep=';', encoding='latin-1', usecols=COLUNAS,
                             dtype={'CO_NCM': str, 'CO_ANO': 'int16', 'CO_MES': 'int8',
                                    'KG_LIQUIDO': str, 'VL_FOB': str}, chunksize=100_000):
        bloco.columns = bloco.columns.str.strip()
        cap = bloco.CO_NCM.astype(str).str.strip().str.startswith('29')
        b = bloco.loc[cap, COLUNAS].copy()
        if b.empty:
            continue
        n_brutos += len(b)
        codigos_iniciais.update(b.CO_NCM.astype(str).str.strip().unique())
        b['KG_LIQUIDO'] = pd.to_numeric(b.KG_LIQUIDO, errors='coerce')
        b['VL_FOB'] = pd.to_numeric(b.VL_FOB, errors='coerce')
        b = b[(b.CO_ANO == ano) & b.CO_MES.between(1, 12)
              & b.KG_LIQUIDO.gt(0) & b.VL_FOB.gt(0)].copy()
        if b.empty:
            continue
        n_validos += len(b)
        meses.update(b.CO_MES.unique())
        partes.append(pd.DataFrame({
            'ncm': b.CO_NCM.astype(str).str.strip(),
            'ano': np.int16(ano),
            'mes': b.CO_MES.astype(np.int8),
            'q': b.KG_LIQUIDO.astype('float64'),
            'fob': b.VL_FOB.astype('float64'),
        }))
        del b, bloco
    if not partes:
        raise ValueError(f'Não há registros válidos em {arquivo.name}.')
    if meses != set(range(1, 13)):
        raise ValueError(f'Arquivo {arquivo.name} sem os 12 meses: {sorted(meses)}')
    dados = pd.concat(partes, ignore_index=True)
    dados['x'] = np.log10(dados.q)
    dados['preco'] = dados.fob / dados.q
    dados['y'] = np.log10(dados.preco)
    contagens = {'periodo': str(ano), 'ncms_iniciais': len(codigos_iniciais),
                 'ncms_validas': dados.ncm.nunique(), 'registros_cap29': n_brutos,
                 'registros_validos': n_validos}
    print(f'{ano}: {n_validos:,} registros válidos; {contagens["ncms_validas"]:,} NCMs.')
    return dados, contagens

Descrições de NCMs do capítulo 29: 2263


## 4. Regressões e agrupamentos

In [4]:
def regressoes(dados):
    linhas = []
    for ncm, g in dados.groupby('ncm', sort=True):
        if len(g) < MIN_OBS or g.x.nunique() < 2:
            continue
        ajuste = stats.linregress(g.x.to_numpy(), g.y.to_numpy())
        r2 = float(ajuste.rvalue ** 2)
        if not np.isfinite(r2):
            continue
        linhas.append({
            'ncm': ncm, 'descricao_ncm': DESCRICOES.get(ncm, ''),
            'n': len(g), 'beta': ajuste.slope, 'alpha': ajuste.intercept,
            'r2': r2, 'log_q_max': float(g.x.max()),
            'sd_log_q': float(g.x.std(ddof=1)),
            'subcapitulo': subcapitulo(ncm),
        })
    return pd.DataFrame(linhas)

def agrupamentos(tabela):
    tab = tabela.copy()
    silhuetas, arvores = [], {}
    for metodo, colunas in [
        ('ward_uni', ['beta']),
        ('ward_multi', ['beta', 'log_q_max', 'sd_log_q']),
    ]:
        tab[metodo] = pd.Series(pd.NA, index=tab.index, dtype='Int64')
        if len(tab) < 5:
            continue
        x = StandardScaler().fit_transform(tab[colunas])
        z = linkage(x, method='ward')
        opcoes = []
        for k in K_CANDIDATOS:
            if k >= len(tab):
                break
            rotulos = cut_tree(z, n_clusters=k).ravel()
            escore = silhouette_score(x, rotulos)
            silhuetas.append({'metodo': metodo, 'k': k, 'silhueta': escore})
            opcoes.append((escore, k, rotulos))
        if not opcoes:
            continue
        _, k, rotulos = max(opcoes, key=lambda item: (item[0], -item[1]))
        tab[metodo] = rotulos + 1
        arvores[metodo] = z
    return tab, pd.DataFrame(silhuetas), arvores

def distancia_corte_ward(z, k):
    """Altura entre fusões que deixa exatamente k grupos no dendrograma."""
    n = z.shape[0] + 1
    if not 1 < int(k) < n:
        return np.nan
    k = int(k)
    return float((z[n-k-1, 2] + z[n-k, 2]) / 2)

def marcar_faixa_corte(ax, z, avaliacoes):
    """Marca cortes testados e o corte associado à maior silhueta média."""
    if avaliacoes.empty:
        return None, np.nan
    avaliacoes = avaliacoes.sort_values('k').copy()
    avaliacoes['k'] = avaliacoes.k.astype(int)
    k_min, k_max = int(avaliacoes.k.min()), int(avaliacoes.k.max())
    cortes = [distancia_corte_ward(z, k) for k in (k_min, k_max)]
    if np.isfinite(cortes).all():
        inferior, superior = min(cortes), max(cortes)
        ax.axhspan(inferior, superior, color='gold', alpha=.18,
                   label=f'Faixa avaliada (k={k_min}–{k_max})', zorder=0)
    melhor = avaliacoes.loc[avaliacoes.silhueta.idxmax()]
    k_escolhido = int(melhor.k)
    escore = float(melhor.silhueta)
    corte = distancia_corte_ward(z, k_escolhido)
    if np.isfinite(corte):
        ax.axhline(corte, color='crimson', linestyle='--', linewidth=1.5,
                   label=f'Maior silhueta: k={k_escolhido} ({escore:.3f})')
    ax.text(.99, .02,
            'k ≥ 3 por critério exploratório',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8,
            bbox={'facecolor':'white', 'alpha':.75, 'edgecolor':'none', 'pad':2})
    ax.legend(frameon=False, fontsize=8, loc='best')
    return k_escolhido, escore

def resumo_grupos(tab):
    saida = []
    for metodo in ('ward_uni', 'ward_multi'):
        for grupo, g in tab.groupby(metodo, dropna=True):
            saida.append({'metodo': metodo, 'grupo': int(grupo), 'n_ncms': len(g),
                          'beta_media': g.beta.mean(), 'beta_mediana': g.beta.median(),
                          'sd_beta': g.beta.std(ddof=1)})
    return pd.DataFrame(saida)

def salvar_csv(df, pasta, nome):
    df.to_csv(pasta / (nome + '.csv'), index=False, encoding='utf-8-sig')

def salvar_grafico(fig, nome, suplementar=False):
    pasta = EXTRAS if suplementar else FIG
    fig.tight_layout()
    fig.savefig(pasta / (nome + '.png'), dpi=220, bbox_inches='tight')
    fig.savefig(pasta / (nome + '.pdf'), bbox_inches='tight')
    plt.close(fig)

def analisar(dados, nome, contagens=None):
    pasta = SAIDA / nome
    pasta.mkdir(exist_ok=True)
    elegiveis = regressoes(dados)
    selecionadas = elegiveis[elegiveis.r2 >= R2_MIN].copy()
    selecionadas, silhuetas, arvores = agrupamentos(selecionadas)
    salvar_csv(elegiveis, pasta, 'ncms_elegiveis')
    salvar_csv(selecionadas, pasta, 'ncms_selecionadas')
    salvar_csv(silhuetas, pasta, 'silhuetas')
    salvar_csv(resumo_grupos(selecionadas), pasta, 'resumo_grupos')
    info = dict(contagens or {})
    info.update({'periodo': nome, 'ncms_elegiveis': len(elegiveis),
                 'ncms_selecionadas': len(selecionadas),
                 'registros_elegiveis': int(elegiveis.n.sum()),
                 'registros_selecionados': int(selecionadas.n.sum()),
                 'beta_media': selecionadas.beta.mean(),
                 'beta_mediana': selecionadas.beta.median(),
                 'beta_dp': selecionadas.beta.std(ddof=1)})
    salvar_csv(pd.DataFrame([info]), pasta, 'resumo')
    graficos_ano(dados, selecionadas, silhuetas, arvores, nome)
    print(nome, '| NCMs elegíveis:', len(elegiveis), '| R² ≥ 0,70:', len(selecionadas))
    return selecionadas, info, arvores

## 5. Gráficos por período


In [5]:
def graficos_ano(dados, tab, silhuetas, arvores, nome):
    if tab.empty:
        return
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(tab.beta, tab.r2, s=16, alpha=.6)
    ax.axhline(R2_MIN, linestyle='--', linewidth=1)
    ax.set(xlabel='Coeficiente β', ylabel='R²', title=f'{nome} — β e R² por NCM')
    salvar_grafico(fig, f'{nome}_beta_r2', True)

    for metodo in ('ward_uni', 'ward_multi'):
        if metodo not in arvores:
            continue
        s = silhuetas[silhuetas.metodo == metodo].copy()
        fig, ax = plt.subplots(figsize=(10, 4.8))
        dendrogram(arvores[metodo], ax=ax, no_labels=True)
        k_escolhido, escore = marcar_faixa_corte(ax, arvores[metodo], s)
        ax.set(title=f'{nome} — {metodo}: corte escolhido por silhueta',
               xlabel='NCMs', ylabel='Distância de Ward')
        salvar_grafico(fig, f'{nome}_dendrograma_{metodo}', True)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(s.k, s.silhueta, marker='o', color='steelblue')
        if k_escolhido is not None:
            ax.axvline(k_escolhido, color='crimson', linestyle='--', linewidth=1,
                       label=f'Maior silhueta: k={k_escolhido} ({escore:.3f})')
            linha = s[s.k.astype(int) == k_escolhido].iloc[0]
            ax.scatter([k_escolhido], [linha.silhueta], color='crimson', zorder=3)
            ax.legend(frameon=False, fontsize=8)
        ax.set(xlabel='Número de grupos', ylabel='Silhueta média',
               title=f'{nome} — escolha de k ({metodo})')
        ax.set_xticks(s.k)
        ax.text(.02, .02, f'Faixa avaliada: k={int(s.k.min())}–{int(s.k.max())}; k ≥ 3 (exploratório)',
                transform=ax.transAxes, ha='left', va='bottom', fontsize=8)
        salvar_grafico(fig, f'{nome}_silhueta_{metodo}', True)
        grupos = [g.beta.to_numpy() for _, g in tab.groupby(metodo)]
        rotulos = [str(i) for i, _ in tab.groupby(metodo)]
        fig, ax = plt.subplots(figsize=(9, 4.6))
        ax.boxplot(grupos, tick_labels=rotulos, showfliers=True)
        ax.set(xlabel='Grupo', ylabel='Coeficiente β',
               title=f'{nome} — dispersão de β ({metodo})')
        salvar_grafico(fig, f'{nome}_beta_grupos_{metodo}', True)

    # Exemplos individuais (pontos de importação e reta estimada).
    presentes = tab.set_index('ncm')
    for ncm, descricao in EXEMPLOS.items():
        if ncm not in presentes.index:
            continue
        g = dados[dados.ncm == ncm]
        linha = presentes.loc[ncm]
        x = np.linspace(g.x.min(), g.x.max(), 100)
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.scatter(g.x, g.y, s=20, alpha=.6, label='Registros de importação')
        ax.plot(x, linha.alpha + linha.beta * x, label='Regressão log-log')
        ax.set(xlabel='log₁₀(quantidade em kg)', ylabel='log₁₀(preço em US$/kg)',
               title=f'{descricao} ({ncm}) — {nome}\nβ={linha.beta:.3f} | R²={linha.r2:.3f}')
        ax.legend(frameon=False)
        salvar_grafico(fig, f'{nome}_exemplo_{ncm}', True)

## 6. Análises anuais e macrobloco


In [6]:
tabelas = {}
resumos = {}
dados_anuais = []

for ano in ANOS:
    dados, cobertura = ler_importacoes(ano)
    tab, resumo, arvore = analisar(dados, str(ano), cobertura)
    tabelas[str(ano)], resumos[str(ano)] = tab, resumo
    # Conserva somente os registros do Capítulo 29 para a etapa consolidada.
    dados_anuais.append(dados)
    del dados, tab, arvore
    gc.collect()

macro = pd.concat(dados_anuais, ignore_index=True)
fatores = {ano: CPI[2025] / CPI[ano] for ano in ANOS}
macro['preco'] = (macro.fob / macro.q) * macro.ano.map(fatores)
macro['y'] = np.log10(macro.preco)
macro['x'] = np.log10(macro.q)

pd.DataFrame({'ano': list(CPI), 'CPI_U': list(CPI.values()),
              'fator_para_2025': [fatores[a] for a in CPI],
              'fonte': FONTE_CPI}).to_csv(SAIDA / 'cpi_utilizado.csv', index=False)

tab, resumo, arvore = analisar(macro, 'macro_2023_2025', {
    'ncms_iniciais': macro.ncm.nunique(),
    'ncms_validas': macro.ncm.nunique(),
    'registros_cap29': len(macro),
    'registros_validos': len(macro),
})
tabelas['macro_2023_2025'], resumos['macro_2023_2025'] = tab, resumo
pd.DataFrame(resumos.values()).to_csv(SAIDA / 'resumo_periodos.csv', index=False,
                                      encoding='utf-8-sig')
del tab, resumo, arvore
gc.collect()

2023: 65,055 registros válidos; 1,093 NCMs.
2023 | NCMs elegíveis: 802 | R² ≥ 0,70: 220
2024: 71,278 registros válidos; 1,103 NCMs.
2024 | NCMs elegíveis: 817 | R² ≥ 0,70: 239
2025: 69,785 registros válidos; 1,109 NCMs.
2025 | NCMs elegíveis: 818 | R² ≥ 0,70: 230
macro_2023_2025 | NCMs elegíveis: 983 | R² ≥ 0,70: 235


44

## 7. Random Forest (80% treino, 20% teste)


In [7]:
def testar_random_forest(tab, nome):
    if len(tab) < 25:
        return []
    base = tab[['ncm', 'beta', 'log_q_max', 'sd_log_q']].dropna().copy()
    y = base.beta.to_numpy()
    x1 = base[['log_q_max', 'sd_log_q']]
    x2 = x1.assign(ncm_num=base.ncm.astype('int64').to_numpy())
    treino, teste = train_test_split(np.arange(len(base)), test_size=.20,
                                     random_state=SEMENTE)
    linhas = []
    for rotulo, x in [('Volume', x1), ('Volume + NCM', x2)]:
        modelo = RandomForestRegressor(n_estimators=RF_ARVORES,
                                      random_state=SEMENTE, n_jobs=1)
        modelo.fit(x.iloc[treino], y[treino])
        for particao, posicoes in [('treino', treino), ('teste', teste)]:
            previsto = modelo.predict(x.iloc[posicoes])
            linhas.append({'periodo': nome, 'modelo': rotulo, 'particao': particao,
                           'n': len(posicoes), 'r2': r2_score(y[posicoes], previsto),
                           'mae_beta': mean_absolute_error(y[posicoes], previsto)})
        previstos = modelo.predict(x.iloc[teste])
        saida = pd.DataFrame({'ncm': base.iloc[teste].ncm.to_numpy(),
                              'beta_observado': y[teste], 'beta_estimado': previstos})
        salvar_csv(saida, SAIDA / nome, 'rf_predicoes_' + rotulo.lower().replace(' + ', '_').replace(' ', '_'))
        fig, ax = plt.subplots(figsize=(5.5, 4.7))
        ax.scatter(y[teste], previstos, s=28, alpha=.7)
        limites = [min(y[teste].min(), previstos.min()), max(y[teste].max(), previstos.max())]
        ax.plot(limites, limites, '--', linewidth=1)
        ax.set(xlabel='β observado', ylabel='β estimado',
               title=f'{nome} — Random Forest: {rotulo} (teste)')
        salvar_grafico(fig, f'{nome}_rf_{rotulo.lower().replace(" + ", "_").replace(" ", "_")}', True)
        del modelo
    return linhas

rf_resultados = []
for nome, tab in tabelas.items():
    rf_resultados.extend(testar_random_forest(tab, nome))

rf = pd.DataFrame(rf_resultados)
salvar_csv(rf, SAIDA, 'resultados_random_forest')
print(rf[rf.particao == 'teste'][['periodo','modelo','r2']].to_string(index=False))

        periodo       modelo        r2
           2023       Volume -0.689961
           2023 Volume + NCM -0.512474
           2024       Volume -0.280001
           2024 Volume + NCM -0.079622
           2025       Volume -0.070450
           2025 Volume + NCM -0.088576
macro_2023_2025       Volume -0.005365
macro_2023_2025 Volume + NCM -0.073536


## 8. Figuras para o TCC


In [8]:
def figuras_principais():
    nomes = [str(y) for y in ANOS]
    t = [tabelas[ano] for ano in nomes]
    macro_tab = tabelas['macro_2023_2025']
    todas = t + [macro_tab]
    labels = nomes + ['Macrobloco']

    # Figura 1: etapas do estudo.
    etapas = ['Importações e dicionário NCM', 'Filtragem do Capítulo 29',
              'Regressões log-log por ano', 'Seleção de NCMs (R² ≥ 0,70)',
              'Agrupamentos Ward e silhueta', 'Comparação entre os anos',
              'Macrobloco corrigido pelo CPI-U', 'Random Forest: 80% / 20%',
              'Sensibilidade com β médio e mediano']
    fig, ax = plt.subplots(figsize=(9, 10))
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
    for j, etapa in enumerate(etapas):
        y = 9.3 - j * 1.03
        ax.text(5, y, etapa, ha='center', va='center',
                bbox={'boxstyle': 'round,pad=0.5', 'facecolor': 'white', 'edgecolor': '0.4'},
                fontsize=11)
        if j < len(etapas)-1:
            ax.annotate('', xy=(5, y-.50), xytext=(5, y-.30),
                        arrowprops={'arrowstyle':'->', 'lw':1.2})
    salvar_grafico(fig, 'Figura_01_Fluxograma')

    # Figura 2: evolução da filtragem.
    colunas = ['ncms_iniciais','ncms_validas','ncms_elegiveis','ncms_selecionadas']
    leg = ['Base inicial', 'Registros válidos', 'Histórico suficiente', 'R² ≥ 0,70']
    pos = np.arange(4)
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, ano in enumerate(nomes):
        valores = [resumos[ano][c] for c in colunas]
        ax.bar(pos+(i-1)*.25, valores, width=.24, label=ano)
    ax.set_xticks(pos, leg)
    ax.set(ylabel='NCMs', title='Filtragem das NCMs por ano')
    ax.legend(frameon=False)
    salvar_grafico(fig, 'Figura_02_Filtragem')

    # Figuras 3 e 4: distribuição de beta.
    fig, ax = plt.subplots(figsize=(9, 5))
    valores = np.concatenate([g.beta.to_numpy() for g in todas])
    limites = np.linspace(valores.min(), valores.max(), 33)
    for label, g in zip(labels, todas):
        ax.hist(g.beta, bins=limites, histtype='step', linewidth=1.8, label=label)
    ax.set(xlabel='Coeficiente β', ylabel='NCMs', title='Distribuição dos coeficientes β')
    ax.legend(frameon=False)
    salvar_grafico(fig, 'Figura_03_Histograma_Beta')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot([g.beta for g in todas], tick_labels=labels)
    ax.set(ylabel='Coeficiente β', title='Comparação dos coeficientes β')
    salvar_grafico(fig, 'Figura_04_Boxplot_Beta')

    # Figura 5: subcapítulos oficiais, 2025.
    grupos = [(cap, g.beta.to_numpy()) for cap, g in t[2].groupby('subcapitulo')]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.boxplot([v for _, v in grupos], tick_labels=[c for c, _ in grupos])
    ax.set(xlabel='Subcapítulo da NCM', ylabel='Coeficiente β',
           title='Elasticidades por subcapítulo — 2025')
    salvar_grafico(fig, 'Figura_05_Subcapitulos_2025')

    # Figura 6: dendrogramas de 2025.
    fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
    for ax, metodo in zip(eixos, ['ward_uni', 'ward_multi']):
        feats = ['beta'] if metodo == 'ward_uni' else ['beta', 'log_q_max', 'sd_log_q']
        x = StandardScaler().fit_transform(t[2][feats])
        z = linkage(x, method='ward')
        dendrogram(z, ax=ax, no_labels=True)
        scores = []
        for k in K_CANDIDATOS:
            if k >= len(t[2]):
                break
            rotulos = cut_tree(z, n_clusters=k).ravel()
            scores.append({'k': k, 'silhueta': silhouette_score(x, rotulos)})
        avaliacoes = pd.DataFrame(scores)
        k_escolhido, escore = marcar_faixa_corte(ax, z, avaliacoes)
        ax.set(title=('Unidimensional' if metodo == 'ward_uni' else 'Multidimensional'),
               xlabel='NCMs', ylabel='Distância de Ward')
    salvar_grafico(fig, 'Figura_06_Dendrogramas_2025')

    # Figura 7: tamanho e dispersão de cada grupo.
    fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
    for ax, metodo in zip(eixos, ['ward_uni','ward_multi']):
        agrup = resumo_grupos(t[2])
        g = agrup[agrup.metodo == metodo].sort_values('grupo')
        ax.barh(g.grupo.astype(str), g.n_ncms)
        ax.invert_yaxis()
        ax.set(xlabel='NCMs', ylabel='Grupo',
               title='Ward unidimensional' if metodo=='ward_uni' else 'Ward multidimensional')
        for j, (_, linha) in enumerate(g.iterrows()):
            dp = '-' if pd.isna(linha.sd_beta) else f'{linha.sd_beta:.3f}'
            ax.text(linha.n_ncms + max(g.n_ncms)*.02, j,
                    f'β={linha.beta_media:.3f} | DP={dp}', va='center', fontsize=8)
        ax.set_xlim(0, max(g.n_ncms)*1.52)
    salvar_grafico(fig, 'Figura_07_Grupos_2025')

    # Figura 8: regressões de quatro substâncias selecionadas em 2025.
    fig, eixos = plt.subplots(2, 2, figsize=(12, 9))
    por_ncm = t[2].set_index('ncm')
    d2025 = dados_anuais[2]
    for ax, (codigo, titulo) in zip(eixos.ravel(), EXEMPLOS.items()):
        if codigo not in por_ncm.index:
            ax.axis('off'); continue
        d = d2025[d2025.ncm == codigo]
        reg = por_ncm.loc[codigo]
        xs = np.linspace(d.x.min(),d.x.max(),100)
        ax.scatter(d.x, d.y, s=18, alpha=.65)
        ax.plot(xs, reg.alpha + reg.beta * xs)
        ax.set(title=f'{titulo} ({codigo})\nβ={reg.beta:.3f}, R²={reg.r2:.3f}',
               xlabel='log₁₀(kg)', ylabel='log₁₀(US$/kg)')
    salvar_grafico(fig, 'Figura_08_Exemplos_Regressoes_2025')

    # Figura 9: variação relativa de preço e percentis empíricos de beta.
    b = macro_tab.beta
    q = np.linspace(1,2,300)
    media, mediana = b.mean(), b.median()
    p005,p025,p975,p995 = b.quantile([.005,.025,.975,.995])
    # Tabelas utilizadas na discussão dos cenários de sensibilidade.
    cenarios = []
    for percentual in (10, 50, 100):
        fator = 1 + percentual / 100
        cenarios.append({
            'aumento_q_pct': percentual,
            'impacto_beta_medio_pct': 100 * (fator**media - 1),
            'impacto_beta_mediano_pct': 100 * (fator**mediana - 1),
            'otimista_95_pct': 100 * (fator**p025 - 1),
            'conservador_95_pct': 100 * (fator**p975 - 1),
            'otimista_99_pct': 100 * (fator**p005 - 1),
            'conservador_99_pct': 100 * (fator**p995 - 1),
        })
    salvar_csv(pd.DataFrame(cenarios), SAIDA, 'tabela_sensibilidade')
    n = len(b)
    erro_padrao = b.std(ddof=1) / np.sqrt(n)
    limites = {'beta_medio': media, 'beta_mediano': mediana,
               'desvio_padrao': b.std(ddof=1),
               'ic95_media_inferior': media - stats.t.ppf(.975, n-1)*erro_padrao,
               'ic95_media_superior': media + stats.t.ppf(.975, n-1)*erro_padrao,
               'ic99_media_inferior': media - stats.t.ppf(.995, n-1)*erro_padrao,
               'ic99_media_superior': media + stats.t.ppf(.995, n-1)*erro_padrao,
               'percentil_005': p005, 'percentil_025': p025,
               'percentil_975': p975, 'percentil_995': p995}
    salvar_csv(pd.DataFrame([limites]), SAIDA, 'estatisticas_beta_macrobloco')

    fig, ax = plt.subplots(figsize=(9,5.3))
    ax.fill_between(q,100*q**p005,100*q**p995,alpha=.12,label='Faixa empírica de 99%')
    ax.fill_between(q,100*q**p025,100*q**p975,alpha=.22,label='Faixa empírica de 95%')
    ax.plot(q,100*q**media,label=f'β médio: {media:.3f}')
    ax.plot(q,100*q**mediana,linestyle='--',label=f'β mediano: {mediana:.3f}')
    ax.set(xlabel='Quantidade relativa (Q₂/Q₁)',ylabel='Preço unitário (base = 100)',
           title='Sensibilidade preço-volume — macrobloco')
    ax.legend(frameon=False)
    salvar_grafico(fig, 'Figura_09_Sensibilidade_Preco')

    # Figura 10: média, mediana e DP nos três anos.
    medias = np.array([g.beta.mean() for g in t])
    medianas = np.array([g.beta.median() for g in t])
    dps = np.array([g.beta.std(ddof=1) for g in t])
    fig, ax = plt.subplots(figsize=(8,5))
    ax.errorbar(ANOS,medias,yerr=dps,marker='o',capsize=5,label='Média ± 1 DP entre NCMs')
    ax.plot(ANOS,medianas,marker='s',linestyle='--',label='Mediana')
    ax.axhline(macro_tab.beta.mean(),linestyle=':',linewidth=1,label='Média macrobloco')
    ax.set_xticks(ANOS)
    ax.set(xlabel='Ano',ylabel='Coeficiente β',title='Evolução das elasticidades centrais')
    ax.legend(frameon=False)
    salvar_grafico(fig, 'Figura_10_Evolucao_Beta')

    # Figura 11: R² de teste do Random Forest.
    rf_teste = rf[rf.particao=='teste'].copy()
    p = rf_teste.pivot(index='periodo', columns='modelo', values='r2').reindex(labels[:-1]+['macro_2023_2025'])
    # A tabela tem rótulo 'macro_2023_2025', não 'Macrobloco'.
    ind = np.arange(len(p)); largura = .35
    fig, ax = plt.subplots(figsize=(8.5,5))
    ax.bar(ind-largura/2,p['Volume'],width=largura,label='Volume')
    ax.bar(ind+largura/2,p['Volume + NCM'],width=largura,label='Volume + NCM')
    ax.axhline(0, linewidth=.8)
    ax.set_xticks(ind, ['2023','2024','2025','Macrobloco'])
    ax.set(ylabel='R² no teste (20%)',title='Random Forest — desempenho fora do treinamento')
    ax.legend(frameon=False)
    salvar_grafico(fig, 'Figura_11_Random_Forest_R2')

    # Gráfico adicional: comparação temporal de compostos identificados.
    linhas = []
    for codigo, titulo in EXEMPLOS.items():
        for ano, tab in zip(ANOS,t):
            g = tab[tab.ncm == codigo]
            if not g.empty:
                linhas.append({'ncm': codigo, 'composto': titulo, 'ano': ano,
                               'beta': g.beta.iloc[0], 'r2': g.r2.iloc[0],
                               'n': int(g.n.iloc[0])})
    exemplos = pd.DataFrame(linhas)
    salvar_csv(exemplos, SAIDA, 'exemplos_compostos')
    fig, ax = plt.subplots(figsize=(9,5))
    for nome, g in exemplos.groupby('composto'):
        ax.plot(g.ano,g.beta,marker='o',label=nome)
    ax.set_xticks(ANOS)
    ax.set(xlabel='Ano',ylabel='Coeficiente β',title='Exemplos de NCMs presentes nos três anos')
    ax.legend(frameon=False)
    salvar_grafico(fig, 'Exemplos_Evolucao_Beta', True)

    # Figura 12: dispersão temporal de beta por NCM no painel balanceado.
    # Exige NCM selecionada nos três anos para comparar desvios-padrão calculados
    # com o mesmo número de estimativas anuais.
    colunas_anos = [str(ano) for ano in ANOS]
    series_beta = [tab.set_index('ncm')['beta'].rename(str(ano))
                   for ano, tab in zip(ANOS, t)]
    painel_beta = pd.concat(series_beta, axis=1).reindex(columns=colunas_anos)
    painel_beta = painel_beta.dropna(subset=colunas_anos)
    painel_beta['dp_beta_2023_2025'] = painel_beta[colunas_anos].std(axis=1, ddof=1)
    painel_beta.index.name = 'ncm'
    salvar_csv(painel_beta.reset_index(), SAIDA,
               'variacao_beta_por_ncm_painel_2023_2025')

    fig, ax = plt.subplots(figsize=(8.5, 5))
    if not painel_beta.empty:
        valores_dp = painel_beta['dp_beta_2023_2025'].to_numpy()
        ax.hist(valores_dp, bins='auto', color='#4C78A8', alpha=.78,
                edgecolor='white')
        media_dp = float(np.mean(valores_dp))
        mediana_dp = float(np.median(valores_dp))
        ax.axvline(media_dp, color='#E45756', linestyle='-', linewidth=1.5,
                   label=f'Média = {media_dp:.3f}')
        ax.axvline(mediana_dp, color='#F2A541', linestyle='--', linewidth=1.5,
                   label=f'Mediana = {mediana_dp:.3f}')
        ax.legend(frameon=False)
        ax.text(.98, .96, f'NCMs presentes nos três anos: {len(painel_beta)}',
                transform=ax.transAxes, ha='right', va='top', fontsize=9)
    else:
        ax.text(.5, .5, 'Não há NCMs selecionadas nos três anos.',
                transform=ax.transAxes, ha='center', va='center')
    ax.set(xlabel='Desvio-padrão de β entre as estimativas anuais',
           ylabel='Número de NCMs',
           title='Variação anual de β por NCM (painel 2023–2025)')
    salvar_grafico(fig, 'Figura_12_Histograma_DP_Beta_por_NCM')

    # Figura 13: interseção das NCMs selecionadas em cada ano (R² ≥ 0,70).
    conjuntos = {str(ano): set(tab.ncm.astype(str))
                 for ano, tab in zip(ANOS, t)}
    a, b, c = (conjuntos[str(ano)] for ano in ANOS)
    regioes_venn = [
        ('Somente 2023', a - b - c),
        ('Somente 2024', b - a - c),
        ('Somente 2025', c - a - b),
        ('2023 e 2024, sem 2025', (a & b) - c),
        ('2023 e 2025, sem 2024', (a & c) - b),
        ('2024 e 2025, sem 2023', (b & c) - a),
        ('Nos três anos', a & b & c),
    ]
    salvar_csv(pd.DataFrame([{'regiao': nome, 'n_ncms': len(itens)}
                             for nome, itens in regioes_venn]),
               SAIDA, 'intersecoes_ncm_selecionadas_2023_2025')

    fig, ax = plt.subplots(figsize=(8, 6.5))
    cores = {'2023': '#5B8FF9', '2024': '#F6BD16', '2025': '#5AD8A6'}
    centros = {'2023': (0, 1), '2024': (1.732, 1), '2025': (.866, -.5)}
    raio = 1.55
    for ano in colunas_anos:
        ax.add_patch(Circle(centros[ano], raio, facecolor=cores[ano],
                            edgecolor=cores[ano], alpha=.28, linewidth=2.0,
                            label=f'{ano}: {len(conjuntos[ano])} NCMs'))
    posicoes = {
        'Somente 2023': (-.72, 1.15),
        'Somente 2024': (2.45, 1.15),
        'Somente 2025': (.866, -1.62),
        '2023 e 2024, sem 2025': (.866, 2.05),
        '2023 e 2025, sem 2024': (.06, .18),
        '2024 e 2025, sem 2023': (1.67, .18),
        'Nos três anos': (.866, .70),
    }
    contagens = dict(regioes_venn)
    for rotulo, pos in posicoes.items():
        ax.text(*pos, str(len(contagens[rotulo])), ha='center', va='center',
                fontsize=10, fontweight='bold', zorder=5)
    ax.set_xlim(-1.75, 3.48)
    ax.set_ylim(-2.35, 3.05)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    ax.set_title('Diagrama de Venn das NCMs selecionadas por ano (R² ≥ 0,70)')
    ax.legend(frameon=False, loc='upper right', bbox_to_anchor=(1.03, 1.0))
    salvar_grafico(fig, 'Figura_13_Venn_NCMs_2023_2025')

figuras_principais()
print('Figuras principais:', len(list(FIG.glob('*.png'))))

Figuras principais: 13


## 9. Consultar exemplos e baixar os resultados


In [9]:
print('\nResumo:')
print(pd.read_csv(SAIDA / 'resumo_periodos.csv')[
    ['periodo','ncms_elegiveis','ncms_selecionadas','beta_media','beta_mediana','beta_dp']
].round(4).to_string(index=False))
print('\nExemplos de compostos:')
print(pd.read_csv(SAIDA / 'exemplos_compostos.csv').round(4).to_string(index=False))

arquivo_zip = PASTA / 'Resultados_TCC_2023_2025.zip'
with zipfile.ZipFile(arquivo_zip, 'w', compression=zipfile.ZIP_DEFLATED,
                     compresslevel=4) as pacote:
    for arquivo in SAIDA.rglob('*'):
        if arquivo.is_file():
            pacote.write(arquivo, arquivo.relative_to(SAIDA))
print('\nResultados salvos em:', arquivo_zip)

try:
    from google.colab import files
    files.download(str(arquivo_zip))
except ImportError:
    print('Fora do Colab, o ZIP está disponível no caminho acima.')


Resumo:
        periodo  ncms_elegiveis  ncms_selecionadas  beta_media  beta_mediana  beta_dp
           2023             802                220     -0.3537       -0.3396   0.1348
           2024             817                239     -0.3665       -0.3461   0.1558
           2025             818                230     -0.3821       -0.3644   0.1477
macro_2023_2025             983                235     -0.3796       -0.3584   0.1609

Exemplos de compostos:
     ncm                composto  ano    beta     r2   n
29051100                 Metanol 2023 -0.2978 0.8044 375
29051100                 Metanol 2024 -0.3111 0.8477 392
29051100                 Metanol 2025 -0.3072 0.8473 447
29071100                   Fenol 2023 -0.3915 0.8830  79
29071100                   Fenol 2024 -0.4349 0.8608  94
29071100                   Fenol 2025 -0.4813 0.9012  86
29032200         Tricloroetileno 2023 -0.4711 0.9635  32
29032200         Tricloroetileno 2024 -0.4955 0.9110  43
29032200         Triclor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# Gráfico 3D dos três grupos do Ward multidimensional — 2025
base_2025_3d = tabelas['2025'].copy()

variaveis = ['beta', 'log_q_max', 'sd_log_q']
X_2025_3d = StandardScaler().fit_transform(base_2025_3d[variaveis])

arvore_2025_3d = linkage(X_2025_3d, method='ward')
grupos_2025_3d = cut_tree(arvore_2025_3d, n_clusters=3).ravel() + 1
silhueta_k3_2025 = silhouette_score(X_2025_3d, grupos_2025_3d)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

cores = ['#4C78A8', '#F58518', '#54A24B']

for grupo in range(1, 4):
    mascara = grupos_2025_3d == grupo
    ax.scatter(
        X_2025_3d[mascara, 0],
        X_2025_3d[mascara, 1],
        X_2025_3d[mascara, 2],
        color=cores[grupo - 1],
        label=f'Grupo {grupo} (n={mascara.sum()})',
        s=38,
        alpha=0.8
    )

ax.set_xlabel('β padronizado')
ax.set_ylabel('log_q_max padronizado')
ax.set_zlabel('sd_log_q padronizado')
ax.set_title(
    f'Ward multidimensional — 2025 (k=3; silhouette={silhueta_k3_2025:.3f})'
)
ax.view_init(elev=24, azim=38)
ax.legend(frameon=False)

salvar_grafico(fig, 'Figura_14_Ward_Multidimensional_3D_2025')

# Salva também o grupo atribuído a cada NCM.
base_2025_3d['ward_multi_k3'] = grupos_2025_3d
salvar_csv(
    base_2025_3d[['ncm', 'ward_multi_k3']],
    SAIDA,
    'grupos_ward_multidimensional_k3_2025'
)
plt.show()
salvar_grafico(fig, 'Figura_14_Ward_Multidimensional_3D_2025')
print('Imagem salva em:', FIG / 'Figura_14_Ward_Multidimensional_3D_2025.png')

Imagem salva em: /content/tcc/resultados/figuras/Figura_14_Ward_Multidimensional_3D_2025.png
